# Parte 1 – Recolección de noticias

## 1.1. Paso 1

**Enunciado**: Configuración inicial: Registrar la fecha de descarga, el medio elegido (A, B o C) y las URLs base en la primera celda Markdown.

Las noticias se descargarán el 16 de octubre de 2026 del diario La República (opción A). Los enlaces a descargar son:

- sitio: https://larepublica.pe
- robots.txt: https://larepublica.pe/robots.txt
- RSS: 
  - https://larepublica.pe/rss/politica.xml
  - https://larepublica.pe/rss/economia.xml
  - https://larepublica.pe/rss/deportes.xml
  - https://larepublica.pe/rss/ciencia.xml

En una primera revisión, con ayuda de Claude, se identificó que solo se iban a recolectar 60 noticias de los últimos días, por lo cual, se tomará la estrategia de rastrear más noticias de los enlaces de cada nota.

Sobre la categoría Ciencia y tecnología, en el diario La República tiene ambas secciones por separado, por lo cual, eventualmente, ambas se agruparán en la categoría ciencia_tecnologia.

## 1.2. Paso 2

**Enunciado**: Extracción: Extraer mínimo 350 noticias con las columnas: titular, bajada, medio, seccion_original, url, fecha. Usar RSS o HTML (requests + BeautifulSoup), implementando pausas entre peticiones y guardando respaldo crudo en datos/raw/.

Para ello, primero se revisará robots.txt.

In [1]:
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; TareaPythonIntermedio/1.0; +https://github.com/Clandra91/Python_intermedio)"}

respuesta = requests.get("https://larepublica.pe/robots.txt", headers=HEADERS, timeout=20)
print("Código:", respuesta.status_code)

for linea in respuesta.text.splitlines():
    if linea.startswith(("Allow", "Disallow", "Crawl-delay")):
        print(linea)

Código: 200
Disallow: /buscador*
Disallow: /data
Disallow: /data/*
Disallow: /contactenos
Disallow: /*/pag/
Disallow: /*/pag/*
Disallow: /*/*/page/*
Disallow: /archive/*
Disallow: /archivo/*
Disallow: /node/*
Disallow: /taxonomy/term/*
Disallow: /assets/*
Disallow: /files/*
Disallow: /comment/reply/*
Disallow: /envivo/*
Disallow: /*/tag/*
Disallow: */*Perú
Disallow: */Encuesta*
Disallow: */EMPRESA/*
Disallow: *ref=photogallery*
Disallow: */sitemaps/*
Disallow: */pf/*
Allow: /sitemap/
Allow: /sitemap/*
Allow: /sitemaps/*
Allow: /rss/
Allow: /rss/*


Se pueden consultar los RSS y los sitemaps, pero no se podrá consultar en el buscador, la paginación de las secciones, encuestas, entre otros.

Las páginas de cada nota (como /economia/2026/10/16/) no están prohibidos, así que se pueden visitar. Además, no hay Crawl-delay, pero se van a agregar pausas.

Ahora, se va a definir la pausa de 1 a 2 segundos (1.5 segundos entre cada intento), además de delimitar máximo 1 reintento (2 pedidos), de acuerdo con lo indicado en la tarea.

In [2]:
import time

PAUSA = 1.5  # segundos de espera después de cada pedido

def descargar(url):
    """Descarga una URL con pausa y, si falla, 1 reintento."""
    for intento in range(2):  # intento 0 = primer pedido; intento 1 = el reintento
        try:
            respuesta = requests.get(url, headers=HEADERS, timeout=20)
            time.sleep(PAUSA)
            if respuesta.status_code == 200:
                return respuesta
            print("Código", respuesta.status_code, "en", url)
        except requests.RequestException as error:
            time.sleep(PAUSA)
            print("Error de conexión:", error)
    return None  # si los 2 intentos fallan, devuelve None


In [3]:
descargar("https://larepublica.pe/rss/ciencia.xml")

<Response [200]>

Ahora, se procede a leer un RSS y guardarlo como respaldo, para hacer una primera prueba con la categoría política.

In [4]:
from datetime import date

fecha_descarga = date.today().isoformat()  # ej. "2026-10-16"

respuesta = descargar("https://larepublica.pe/rss/politica.xml")

# Se guarda el archivo tal como llegó (respaldo crudo)
ruta = f"datos/raw/rss_politica_{fecha_descarga}.xml"
with open(ruta, "wb") as archivo:
    archivo.write(respuesta.content)
print("Guardado:", ruta)


Guardado: datos/raw/rss_politica_2026-10-16.xml


In [5]:
from bs4 import BeautifulSoup

sopa = BeautifulSoup(respuesta.content, "xml")
items = sopa.find_all("item")
print("Noticias en el feed:", len(items))
items[0]


Noticias en el feed: 28


<item>
<title>
                                 Alejandro Aguinaga viajó a Argentina el martes 13 de octubre y todavía no regresa pese a debate por facultades 
                            </title>
<link>https://larepublica.pe/politica/2026/10/16/alejandro-aguinaga-viajo-a-argentina-el-martes-13-de-octubre-y-todavia-no-regresa-pese-a-debate-por-facultades-hnews-512448</link>
<guid isPermaLink="true">https://larepublica.pe/politica/2026/10/16/alejandro-aguinaga-viajo-a-argentina-el-martes-13-de-octubre-y-todavia-no-regresa-pese-a-debate-por-facultades-hnews-512448</guid>
<description>  
                             Según su registro en Migraciones, su salida del país fue el 13 de octubre, en plena discusión del pedido de facultades legislativas del Ejecutivo. Es su tercer viaje al exterior este año, tras Panamá en enero y Estados Unidos en agosto. 
                            </description>
<image:image>
<image:loc>https://larepublica.cronosmedia.glr.pe/original/2026/10/16/6ad2dde6c07d25

Para poder visualizar mejor el resultado se pasará a una tabla.

In [6]:
import pandas as pd

filas = []  # Se crea una lista vacía, donde irá una noticia por elemento

for item in items:
    url = item.find("link").get_text(strip=True)

    # Se arma un diccionario con las 6 columnas de esta noticia
    noticia = {
        "titular": item.find("title").get_text(strip=True),
        "bajada": item.find("description").get_text(strip=True),
        "medio": "La Republica",
        "seccion_original": url.split("/")[3],
        "url": url,
        "fecha": "-".join(url.split("/")[4:7]),
    }
    filas.append(noticia)  # Se agrega la noticia a la lista

rss_politica = pd.DataFrame(filas)
print(rss_politica.shape)
rss_politica.head()

(28, 6)


,titular,bajada,medio,seccion_original,url,fecha
0,Alejandro Aguinaga viajó a Argentina el martes...,"Según su registro en Migraciones, su salida de...",La Republica,politica,https://larepublica.pe/politica/2026/10/16/ale...,2026-10-16
1,Álvaro Vargas Llosa niega intromisión política...,Vargas Llosa afirma que la soberanía peruana n...,La Republica,politica,https://larepublica.pe/politica/2026/10/16/alv...,2026-10-16
2,Solicitan la destitución de Roberto Burneo por...,"La asociación civil Iniciativa Democrática, or...",La Republica,politica,https://larepublica.pe/politica/2026/10/16/sol...,2026-10-16
3,Puno: alcalde de Quiaca muere tras caer a un b...,El cuerpo de Ricardo Quispe presentaba múltipl...,La Republica,politica,https://larepublica.pe/politica/2026/10/16/pun...,2026-10-16
4,Ministro Rafael Belaúnde propone que las FF.AA...,El titular del Ministerio de Defensa explicó q...,La Republica,politica,https://larepublica.pe/politica/2026/10/16/min...,2026-10-16


Como se observa, se obtuvieron 28 noticias, y se logró completar las  columnas requeridas. 

Además, como se indicó previamente, debido a que no se logrará recabar la cantidad de noticias requeridas para esta tarea buscando solamente a través del RSS, se procederá a rastrear más noticias a través de los enlaces. Por ello, la información de las fechas y sección (política, economía, etc.), se extraerán del enlace (URL).

Luego de realizar la prueba, se procede a repetirlo con las 4 secciones: política, economía, deportes, ciencia.

In [ ]:
SECCIONES_RSS = ["politica", "economia", "deportes", "ciencia"]

def leer_rss(seccion):
    """Descarga el RSS de una sección, guarda el respaldo crudo y devuelve una tabla."""
    url_rss = f"https://larepublica.pe/rss/{seccion}.xml"
    respuesta = descargar(url_rss)
    if respuesta is None:
        print("No se pudo descargar:", url_rss)
        return pd.DataFrame()

    # Respaldo crudo
    with open(f"datos/raw/rss_{seccion}_{fecha_descarga}.xml", "wb") as archivo:
        archivo.write(respuesta.content)

    # Lectura de las noticias (igual que con política)
    sopa = BeautifulSoup(respuesta.content, "xml")
    filas = []
    for item in sopa.find_all("item"):
        url = item.find("link").get_text(strip=True)
        filas.append({
            "titular": item.find("title").get_text(strip=True),
            "bajada": item.find("description").get_text(strip=True),
            "medio": "La Republica",
            "seccion_original": url.split("/")[3],
            "url": url,
            "fecha": "-".join(url.split("/")[4:7]),
        })
    return pd.DataFrame(filas)

In [8]:
tablas = []
for seccion in SECCIONES_RSS:
    tabla = leer_rss(seccion)
    print(seccion, "->", len(tabla), "noticias")
    tablas.append(tabla)

rss = pd.concat(tablas, ignore_index=True)
print("Total desde RSS:", rss.shape)
rss["seccion_original"].value_counts()

politica -> 28 noticias
economia -> 13 noticias
deportes -> 11 noticias
ciencia -> 7 noticias
Total desde RSS: (59, 6)


seccion_original
politica    28
economia    13
deportes    11
ciencia      7
Name: count, dtype: int64

De acuerdo con lo que había anticipado Claude, no va ser posible llegar a la cantidad de noticias que requiere la Tarea 1 (350), ya que solo se han obtenido 59 noticias. Por ello, se va recurrir a leer los enlaces que hay en las 59 noticias identificadas.